In [ ]:
# %% [0] Geometry, Mesh & Material Configuration  --  DANUBE (O-band push-pull, one arm)
#
#   One arm of the GSG push-pull modulator, from the Istanbul notebook with 60 deg slab ends.
#   The right domain edge is the signal centre (mirror plane, inside the gold): exact for one arm.
#   Four cross-sections, chosen with TEE and OE:
#     TEE = False: outside of tee, 28 um x 2 um ground   | TEE = True: inside of tee, 2.0 x 0.5 um ground stub
#     OE  = 0.0  : standard BOX                         | OE  = 0.7 : BOX over-etched 700 nm outside the slab
#   SLAB_W is the slab TOP width; the ground stub (inside of tee) does not move with SLAB_W.
import warnings
from collections import OrderedDict
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from shapely.affinity import scale as _scale
from shapely.geometry import Polygon, MultiPolygon, box
from shapely.ops import unary_union
from skfem import MeshTri
from femwell.mesh import mesh_from_OrderedDict

warnings.filterwarnings("ignore")

# ------------------------------------------------------------------------------
# femwell bug fix.  mesh_from_OrderedDict() tries to switch off gmsh's
# "extend mesh size from boundary" with `gmsh.model.mesh.MeshSizeExtendFromBoundary = 0`,
# which only sets a Python attribute: the gmsh option is never changed.  This
# wrapper applies the options femwell intended, so the mesh follows `resolutions`.
import gmsh
if not getattr(gmsh.model.mesh.field, "_size_fix", False):
    _orig_set_bg = gmsh.model.mesh.field.setAsBackgroundMesh

    def _set_bg_fixed(tag):
        _orig_set_bg(tag)
        gmsh.option.setNumber("Mesh.MeshSizeExtendFromBoundary", 0)
        gmsh.option.setNumber("Mesh.MeshSizeFromPoints", 0)
        gmsh.option.setNumber("Mesh.MeshSizeFromCurvature", 0)

    gmsh.model.mesh.field.setAsBackgroundMesh = _set_bg_fixed
    gmsh.model.mesh.field._size_fix = True

# ============================================================
# 1. FIXED PHYSICAL & GEOMETRIC PARAMETERS
# ============================================================
wl = 1.310          # um (O band)
wl_m = wl * 1e-6
k0 = 2.0 * np.pi / wl               # 1/um
k0_m = k0 * 1e6                     # 1/m
L_cm = 1.0          # cm
V_bias = 1.0        # V
r33 = 30.8e-12      # m/V
n_guess = 1.90
num_modes_search = 12

# ---- Danube cross-section selectors -------------------------------------------
TEE = False         # False: outside of tee (28 um ground, 2 um gold)
                    # True : inside of tee (ground = 2.0 x 0.5 um stub on the slab)
OE = 0.0            # SiO2 BOX over-etch depth (um) everywhere outside the LN slab
                    # footprint, vertical walls; 0.0 = standard, 0.7 = over-etched

# Geometry Dimensions (um), y = 0 at the BOX / LN interface.  One arm of the
# push-pull GSG modulator: the structure is mirror-symmetric about the signal
# centre, which lies inside the gold, so the right domain edge is placed there.
BOX_H = 4.7
TECN = 0.450
WG_H = 0.225
SLAB_H = TECN - WG_H                # 0.225 um LN slab
WG_TOP = 0.8
ALPHA = 60.0                        # rib sidewall angle (deg)
ALPHA_SLAB = 60.0                   # slab-end sidewall angle (deg): the process value
GAP = 4.15
SIG_W = 10.0                        # full signal width (both arms) -> SIG_W / 2 per arm
GND_W = 28.0                        # outside of tee: ground width
STUB_W = 2.0                        # inside of tee: ground stub width (from the gap edge)
STUB_H = 0.5                        # inside of tee: ground stub thickness (on the slab top)
MTX = 2.0                           # gold thickness (conformal)
MARGIN = 5.0                        # vacuum/BOX beyond the ground (outside of tee) edge
CLAD_H = 15.0                       # air above the slab
CAP_W = 2.0
CAP_H = 1.40
SLAB_W = 10.15                      # LN slab TOP width (bottom wider by 2 SLAB_H / tan(ALPHA_SLAB))

basetta = WG_H / np.tan(np.deg2rad(ALPHA))
WG_BOTTOM = WG_TOP + 2.0 * basetta
SLAB_DX = SLAB_H / np.tan(np.deg2rad(ALPHA_SLAB))
X_R = round(GAP / 2.0 + SIG_W / 2.0, 9)            # right domain edge: signal centre (symmetry plane)
X_L = round(-(GAP / 2.0 + GND_W + MARGIN), 9)      # left domain edge (same for both tee cases)
DEV_W = X_R - X_L
assert CAP_W > WG_BOTTOM and GAP > CAP_W and SLAB_W > GAP, "rib / cap / gap / slab inconsistent"
assert SLAB_W / 2.0 + SLAB_DX < X_R - 0.3, "slab must end under the signal, before the symmetry plane"
SLAB_EXT = (SLAB_W - GAP) / 2.0                    # slab top edge beyond the gap edge (per side)

# ---- Material dispersion: every optical constant is computed at `wl` ---------
def n_LN(lam):
    """Congruent LiNbO3, Zelmon et al., JOSA B 14, 3319 (1997).  Returns (ne, no)."""
    l2 = lam ** 2
    ne_ = np.sqrt(1 + 2.9804 * l2 / (l2 - 0.02047) + 0.5981 * l2 / (l2 - 0.0666)
                  + 8.9543 * l2 / (l2 - 416.08))
    no_ = np.sqrt(1 + 2.6734 * l2 / (l2 - 0.01764) + 1.2290 * l2 / (l2 - 0.05914)
                  + 12.614 * l2 / (l2 - 474.6))
    return ne_, no_


def n_SiO2(lam):
    """Fused silica, Malitson, JOSA 55, 1205 (1965)."""
    l2 = lam ** 2
    return np.sqrt(1 + 0.6961663 * l2 / (l2 - 0.0684043 ** 2)
                   + 0.4079426 * l2 / (l2 - 0.1162414 ** 2)
                   + 0.8974794 * l2 / (l2 - 9.896161 ** 2))


# Gold: Johnson & Christy, PRB 6, 4370 (1972), (lambda um, n, k); eps = (n - i k)^2
# interpolated linearly in photon energy.  exp(+jwt): Im(eps) < 0 is loss.
_AU_JC = np.array([
    (0.617, 0.21, 3.272), (0.659, 0.14, 3.697), (0.705, 0.13, 4.103),
    (0.756, 0.14, 4.542), (0.821, 0.16, 5.083), (0.892, 0.17, 5.663),
    (0.984, 0.22, 6.350), (1.088, 0.27, 7.150), (1.216, 0.35, 8.145),
    (1.393, 0.43, 9.519), (1.610, 0.56, 11.210), (1.937, 0.92, 13.780)])


def eps_Au_JC(lam):
    lam_t, n_t, k_t = _AU_JC.T
    assert lam_t.min() <= lam <= lam_t.max(), "wavelength outside the J&C table"
    E_t = 1.239842 / lam_t
    eps_t = (n_t - 1j * k_t) ** 2
    E = 1.239842 / lam
    return complex(np.interp(E, E_t[::-1], eps_t.real[::-1]),
                   np.interp(E, E_t[::-1], eps_t.imag[::-1]))


ne, no = n_LN(wl)
n_sio2 = n_SiO2(wl)
eps_au = eps_Au_JC(wl)
print(f"[materials @ {wl*1e3:.0f} nm] LN ne={ne:.6f} no={no:.6f} | "
      f"SiO2 n={n_sio2:.6f} | Au eps={eps_au.real:.2f}{eps_au.imag:+.2f}j")

# ---- Mesh scales (same as the etched lifted cell) -----------------------------
h_core = 0.040
h_skin = 0.012                      # gold on LN: the SPP decays into Au in ~22 nm
SKIN_T = 0.070                      # gold shell resolving the ~23 nm optical skin depth
BOT_SKIN_L = 0.5                    # shell on gold-on-BOX beyond an etched slab end
OUTER_WALL_SKIN_H = 0.5
CORNER_R = 0.050                    # corner patch half-size (um): metal/LN corners
h_corner = 0.006                    # mesh size inside the corner patches (um)

MATERIALS = {
    "LN":     {"color": "#8b95a1", "eps_dc": (28.0, 44.0), "eps_opt": (ne**2, no**2, no**2), "desc": "LN Core / Slab"},
    "SiO2":   {"color": "#5aa9f0", "eps_dc": (3.75, 3.75), "eps_opt": (n_sio2**2, n_sio2**2, n_sio2**2), "desc": "SiO2 Cap / Underclad"},
    "air":    {"color": "#ffffff", "eps_dc": (1.00, 1.00), "eps_opt": (1.00, 1.00, 1.00), "desc": "Air Cladding"},
    "Au_sig": {"color": "#f6c453", "eps_dc": (1.00, 1.00), "eps_opt": (eps_au, eps_au, eps_au), "desc": "Signal Electrode (+1V)"},
    "Au_gnd": {"color": "#e5a92e", "eps_dc": (1.00, 1.00), "eps_opt": (eps_au, eps_au, eps_au), "desc": "Ground Electrode (0V)"},
}

PREFIX_TO_MAT = [
    ("core", "LN"), ("cap", "SiO2"), ("slab", "LN"), ("box", "SiO2"),
    ("clad", "air"), ("elR", "Au_sig"), ("elL", "Au_gnd"),
]

def material_of(region_name):
    base = region_name.split("___")[0]
    for prefix, mat in PREFIX_TO_MAT:
        if base.startswith(prefix):
            return mat
    raise KeyError(f"Unknown material for region: {region_name}")

def mirror(p):
    return _scale(p, xfact=-1.0, yfact=1.0, origin=(0, 0))

def _areal(g, tol=1e-10):
    """Keep only the polygonal part of a shapely result (drops touching lines/points)."""
    if g.is_empty:
        return Polygon()
    if isinstance(g, Polygon):
        return g if g.area > tol else Polygon()
    parts = [p for p in getattr(g, "geoms", []) if isinstance(p, (Polygon, MultiPolygon))]
    parts = [q for p in parts for q in (p.geoms if isinstance(p, MultiPolygon) else [p])
             if q.area > tol]
    if not parts:
        return Polygon()
    return parts[0] if len(parts) == 1 else MultiPolygon(parts)

# ============================================================
# 2. GEOMETRY GENERATOR
# ============================================================
def build_polygons():
    r9 = lambda v: round(float(v), 9)
    xg = r9(GAP / 2.0)                              # gap edges at -xg (ground) and +xg (signal)
    xs = r9(SLAB_W / 2.0)                           # slab edge at the slab TOP
    xsb = r9(xs + SLAB_DX)                          # slab edge at the BOX
    x_gout = r9(-(xg + (STUB_W if TEE else GND_W))) # ground outer edge
    t_gnd = STUB_H if TEE else MTX
    x_fine = r9(xsb + 1.0)                          # fine slab / recess mesh out to |x| = xsb + 1

    y_slab_top = r9(SLAB_H)
    y_cap_top = r9(SLAB_H + CAP_H)
    y_clad_top = r9(SLAB_H + CLAD_H)
    y_fine_top = r9(SLAB_H + 0.6)
    y_rec = r9(-OE)

    # ---- 1. LN: rib + slab with 60 deg ends ----------------------------------
    core_poly = Polygon([
        (-WG_BOTTOM / 2.0, y_slab_top), (-WG_TOP / 2.0, y_slab_top + WG_H),
        (WG_TOP / 2.0, y_slab_top + WG_H), (WG_BOTTOM / 2.0, y_slab_top)])
    slab = Polygon([(-xsb, 0.0), (xsb, 0.0), (xs, y_slab_top), (-xs, y_slab_top)])

    # ---- 2. SiO2 cap ---------------------------------------------------------
    cap_poly = box(-CAP_W / 2.0, y_slab_top, CAP_W / 2.0, y_fine_top).difference(core_poly)
    cap_up = box(-CAP_W / 2.0, y_fine_top, CAP_W / 2.0, y_cap_top)

    # ---- 3. BOX, over-etched by OE outside the slab footprint (vertical walls)
    box_all = box(X_L, -BOX_H, X_R, 0.0)
    if OE > 0:
        box_all = box_all.difference(unary_union([box(X_L, y_rec, -xsb, 0.0), box(xsb, y_rec, X_R, 0.0)]))

    # ---- 4. Conformal gold of thickness t over [x1, x2]: on the slab top
    #         (y up to SLAB_H + t inside the slab footprint), on the BOX (up to t)
    #         outside it, and filling the over-etch recess below it.
    def gold(x1, x2, t):
        parts = [box(x1, 0.0, x2, t).difference(box(-xsb, 0.0, xsb, 1e3))]
        lo, hi = max(x1, -xsb), min(x2, xsb)
        if hi > lo:
            parts.append(box(lo, 0.0, hi, r9(y_slab_top + t)))
        if OE > 0:
            parts.append(box(x1, y_rec, x2, 0.0).difference(box(-xsb, y_rec, xsb, 0.0)))
        return _areal(unary_union(parts).difference(slab))

    el_sig = gold(xg, X_R, MTX)                     # signal, out to the symmetry plane
    el_gnd = gold(x_gout, -xg, t_gnd)               # ground (28 um) or stub (2 um x 0.5 um)
    for _e in (el_sig, el_gnd):
        assert _e.geom_type == "Polygon", "electrode is not a single body"

    # ---- 5. Air: everything else (above the BOX and in un-metallised recesses)
    solids = unary_union([core_poly, slab, cap_poly, cap_up, el_sig, el_gnd, box_all])
    clad_all = box(X_L, -BOX_H, X_R, y_clad_top).difference(solids)
    clad_gap = _areal(clad_all.intersection(box(-xg, 0.0, xg, y_fine_top)))
    clad_gap_up = _areal(clad_all.intersection(box(-xg, y_fine_top, xg, y_cap_top + 0.6)))
    clad_near = _areal(clad_all.intersection(box(-x_fine, y_rec - 1e-9 if OE > 0 else 0.0, -xg, y_fine_top)))
    clad_far = _areal(clad_all.difference(box(-xg, 0.0, xg, y_cap_top + 0.6))
                      .difference(box(-x_fine, y_rec - 1e-9 if OE > 0 else 0.0, -xg, y_fine_top)))

    xb1 = xg + 1.0
    base = OrderedDict()
    base["core"] = core_poly
    base["cap"] = cap_poly
    base["cap_up"] = cap_up
    base["elR_bulk"] = el_sig
    base["elL_bulk"] = el_gnd
    base["slab_fine"] = slab
    base["box_near"] = _areal(box_all.intersection(box(-xb1, -0.5, xb1, 0.0)))
    base["box_mid"] = _areal(box_all.intersection(box(-x_fine, -1.5, x_fine, 0.0)).difference(box(-xb1, -0.5, xb1, 0.0)))
    base["box_far"] = _areal(box_all.difference(box(-x_fine, -1.5, x_fine, 0.0)))
    base["clad_gap"] = clad_gap
    base["clad_gap_up"] = clad_gap_up
    base["clad_near"] = clad_near
    base["clad_far"] = clad_far

    # ---- 6. Refinement ----------------------------------------------------------
    # 6a. corner patches: gold / LN corners, slab ends, stub end, recess corners
    pts = [(-xg, y_slab_top), (xg, y_slab_top),                 # gold inner corners on the slab
           (xs, y_slab_top), (xsb, 0.0)]                        # signal-side slab end (in gold)
    pts += [(-xs, y_slab_top), (-xsb, 0.0)]                     # ground-side slab end (gold or air)
    if TEE:
        pts.append((x_gout, y_slab_top) if -x_gout < xs else (x_gout, 0.0))   # stub end
    if OE > 0:
        pts += [(xsb, y_rec), (-xsb, y_rec)]                    # pedestal feet
    pts = list(dict.fromkeys((r9(px), r9(py)) for px, py in pts))
    r_c = CORNER_R
    edges = [xb1, -xb1, x_fine, -x_fine, CAP_W / 2.0, -CAP_W / 2.0]
    for i, (px, py) in enumerate(pts):
        for qx, qy in pts[i + 1:]:
            r_c = min(r_c, max(abs(px - qx), abs(py - qy)) / 2.5)
        for ex in edges:
            if 1e-9 < abs(px - ex) < r_c + 1e-9:
                r_c = 0.8 * abs(px - ex)
    assert r_c >= 0.005, f"corner patches would be {r_c*1e3:.1f} nm: not meshable"
    patches = unary_union([box(px - r_c, py - r_c, px + r_c, py + r_c) for px, py in pts])

    # 6b. gold skin: gold within SKIN_T of the slab, of the BOX / pedestal surface
    #     out to BOT_SKIN_L past each slab end, and the inner walls (full height)
    support = unary_union([slab, box_all])
    skin_zone = unary_union([
        slab.buffer(SKIN_T, join_style=2),
        support.buffer(SKIN_T, join_style=2).intersection(
            box(-xsb - BOT_SKIN_L, -BOX_H, xsb + BOT_SKIN_L, y_slab_top)),
        box(xg, 0.0, xg + SKIN_T, y_clad_top), box(-xg - SKIN_T, 0.0, -xg, y_clad_top),
        box(x_gout, 0.0, x_gout + SKIN_T, y_slab_top + OUTER_WALL_SKIN_H),
    ])

    polys = OrderedDict()
    for name, reg in base.items():
        if reg.is_empty:
            continue
        pc = _areal(reg.intersection(patches))
        if not pc.is_empty:
            polys[name + "_corner"] = pc
        reg = _areal(reg.difference(patches))
        if name.startswith("el"):
            sk = _areal(reg.intersection(skin_zone))
            if not sk.is_empty:
                polys[name.replace("bulk", "skin")] = sk
            reg = _areal(reg.difference(skin_zone))
        if not reg.is_empty:
            polys[name] = reg
    return polys, y_clad_top

# ============================================================
# 3. MESH GENERATION & RESOLUTION SETUP
# ============================================================
polygons, Y_TOP = build_polygons()

_base_res = {
    "core":        (h_core, 0.30),
    "cap":         (0.035, 0.30),
    "cap_up":      (0.080, 0.30),
    "clad_gap":    (0.040, 0.30),
    "clad_gap_up": (0.100, 0.30),
    "slab_fine":   (0.025, 0.30),
    "clad_near":   (0.050, 0.30),
    "box_near":    (0.040, 0.30),
    "box_mid":     (0.100, 0.30),
    "box_far":     (0.800, 1.00),
    "elR_bulk":    (0.800, 0.50),
    "elL_bulk":    (0.800, 0.50),
    "elR_skin":    (h_skin, 0.20),
    "elL_skin":    (h_skin, 0.20),
    "clad_far":    (0.800, 1.00),
}
resolutions = {}
for _k in polygons:
    if _k.endswith("_corner"):
        resolutions[_k] = {"resolution": h_corner, "distance": 0.10}
    else:
        _r, _d = _base_res[_k]
        resolutions[_k] = {"resolution": _r, "distance": _d}

# ---- Validity / consistency assertions ---------------------------------------
_names = list(polygons)
for _n, _p in polygons.items():
    assert _p.is_valid and not _p.is_empty and _p.area > 1e-12, f"degenerate region {_n}"
for _i in range(len(_names)):
    for _j in range(_i + 1, len(_names)):
        _ov = polygons[_names[_i]].intersection(polygons[_names[_j]]).area
        assert _ov < 1e-10, f"overlap {_names[_i]} & {_names[_j]}: {_ov:.2e}"
_tot = sum(p.area for p in polygons.values())
assert abs(_tot - DEV_W * (BOX_H + Y_TOP)) < 1e-7, "tiling gap"

print(f"[geometry OK] {len(polygons)} regions tile exactly")
print(f"              Danube {'INSIDE' if TEE else 'OUTSIDE'} of tee | over-etch {OE*1e3:.0f} nm | slab top {SLAB_W:.2f} um "
      f"(edge {SLAB_EXT:+.2f} um past each gap edge) | gap {GAP:.2f} um")
print(f"              ground: {'stub %.1f x %.1f um' % (STUB_W, STUB_H) if TEE else '%.0f x %.1f um' % (GND_W, MTX)} | "
      f"signal {SIG_W/2:.1f} um per arm (domain edge at the symmetry plane x = {X_R:.3f})")

raw_mesh = mesh_from_OrderedDict(
    polygons,
    resolutions=resolutions,
    default_resolution_min=0.5 * min(h_skin, h_corner),
    default_resolution_max=0.6,
)
skfem_mesh = MeshTri(raw_mesh.points[:, :2].T, raw_mesh.cells_dict["triangle"].T)
tri_subdomains = raw_mesh.cell_data_dict["gmsh:physical"]["triangle"]
name_to_id = {
    name: data[0] if hasattr(data, "__getitem__") else data
    for name, data in raw_mesh.field_data.items()
}

mat_of_el = np.empty(skfem_mesh.nelements, dtype=object)
for raw_name, s_id in name_to_id.items():
    if raw_name.split("___")[0] in polygons:
        mat_of_el[tri_subdomains == s_id] = material_of(raw_name)

assert not np.any(mat_of_el == None), (  # noqa: E711
    f"{int(np.sum(mat_of_el == None))} unassigned elements")  # noqa: E711
print(f"[mesh OK] {skfem_mesh.nelements} triangles, {skfem_mesh.nvertices} vertices")

print("=" * 84)
print(f"{'CANONICAL MATERIAL':<28} | {'DC EPS (x, y)':<16} | {'OPTICAL EPS (xx, yy, zz)':<28}")
print("-" * 84)
for key, data in MATERIALS.items():
    dc_s = f"({data['eps_dc'][0]:.1f}, {data['eps_dc'][1]:.1f})"
    opt_val = data['eps_opt'][0]
    opt_s = f"({opt_val.real:.1f}{opt_val.imag:+.1f}j)" if np.iscomplex(opt_val) else f"({opt_val.real:.3f})"
    print(f"{data['desc']:<28} | {dc_s:<16} | {opt_s:<28}")
print("=" * 84)

# ---- Cross-section WITHOUT the mesh (material the solver assigns to each triangle)
from matplotlib.colors import ListedColormap

_mkeys = list(MATERIALS)
_midx = np.array([_mkeys.index(m) for m in mat_of_el], dtype=float)
_mcmap = ListedColormap([MATERIALS[m]["color"] for m in _mkeys])
_by_mat = {}
for _n, _p in polygons.items():
    _by_mat.setdefault(material_of(_n), []).append(_p)
_mat_shapes = {m: unary_union(v) for m, v in _by_mat.items()}


def _draw_materials(ax):
    ax.tripcolor(skfem_mesh.p[0], skfem_mesh.p[1], skfem_mesh.t.T, facecolors=_midx,
                 cmap=_mcmap, vmin=-0.5, vmax=len(_mkeys) - 0.5,
                 edgecolors="face", linewidth=0.0, antialiased=False)
    for _shape in _mat_shapes.values():
        for _part in (_shape.geoms if hasattr(_shape, "geoms") else [_shape]):
            for _ring in [_part.exterior, *_part.interiors]:
                _xy = np.asarray(_ring.coords)
                ax.plot(_xy[:, 0], _xy[:, 1], color="k", lw=0.6)


_case = f"{'inside' if TEE else 'outside'} of tee, {'over-etched ' + format(OE * 1e3, '.0f') + ' nm' if OE > 0 else 'standard'}"
fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(13, 7.4), gridspec_kw=dict(height_ratios=[1, 1.2]))
_draw_materials(ax_a)
ax_a.legend(handles=[mpatches.Patch(color=MATERIALS[m]["color"], label=f"{m}: {MATERIALS[m]['desc']}")
                     for m in _mkeys if m in _mat_shapes], loc="upper left", framealpha=0.95, fontsize=8)
ax_a.set_xlim(X_L, X_R); ax_a.set_ylim(-2.0, 4.0); ax_a.set_aspect("equal")
ax_a.set_title(f"Danube ({_case}), one arm, full domain width | right edge = signal centre (symmetry plane)")
_draw_materials(ax_b)
ax_b.set_xlim(-SLAB_W / 2.0 - 1.5, X_R); ax_b.set_ylim(-1.2, 2.6); ax_b.set_aspect("equal")
ax_b.set_xlabel(r"$x$ ($\mu$m)"); ax_b.set_ylabel(r"$y$ ($\mu$m)")
ax_b.set_title(rf"Zoom | slab top width {SLAB_W:.2f} $\mu$m, gap {GAP:.2f} $\mu$m, $\lambda$ = {wl*1e3:.0f} nm")
plt.tight_layout()
plt.show()

# ---- Cross-section with the mesh ----------------------------------------------
fig, ax = plt.subplots(figsize=(13, 4.5))
_draw_materials(ax)
skfem_mesh.draw(ax=ax, color="black", lw=0.15)
ax.set_xlim(-SLAB_W / 2.0 - 1.5, X_R); ax.set_ylim(-1.2, 2.6); ax.set_aspect("equal")
ax.set_title(f"Danube ({_case}) mesh | {skfem_mesh.nelements} triangles")
plt.tight_layout()
plt.show()

In [ ]:
# %% [1] Anisotropic DC Electrostatics
from skfem import Basis, ElementTriP0, ElementTriP1, BilinearForm, asm, condense, solve
from skfem.helpers import grad

basis_dc = Basis(skfem_mesh, ElementTriP1(), intorder=4)
basis_p0 = Basis(skfem_mesh, ElementTriP0(), intorder=4)

eps_dc_x = np.array([MATERIALS[m]["eps_dc"][0] for m in mat_of_el], dtype=float)
eps_dc_y = np.array([MATERIALS[m]["eps_dc"][1] for m in mat_of_el], dtype=float)

@BilinearForm
def laplace_aniso(u, v, w):
    return w.eps_x * grad(u)[0] * grad(v)[0] + w.eps_y * grad(u)[1] * grad(v)[1]

K = asm(
    laplace_aniso,
    basis_dc,
    eps_x=basis_p0.interpolate(eps_dc_x),
    eps_y=basis_p0.interpolate(eps_dc_y),
)

sig_elements = np.where(mat_of_el == "Au_sig")[0]
gnd_elements = np.where(mat_of_el == "Au_gnd")[0]

dofs_signal = basis_dc.get_dofs(elements=sig_elements).all()
dofs_ground = basis_dc.get_dofs(elements=gnd_elements).all()
dofs_dirichlet = np.unique(np.concatenate([dofs_signal, dofs_ground]))

u_dirichlet = np.zeros(basis_dc.N)
u_dirichlet[dofs_signal] = V_bias
u_dirichlet[dofs_ground] = 0.0

K_c, f_c, u_c, I = condense(K, np.zeros(basis_dc.N), x=u_dirichlet, D=dofs_dirichlet)
potential_nodes = u_c.copy()
potential_nodes[I] = solve(K_c, f_c)

grad_V = basis_dc.interpolate(potential_nodes).grad
Ex_dc = -np.mean(grad_V[0], axis=-1) if grad_V.ndim == 3 else -grad_V[0]

# Diagnostic plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

ax0 = axes[0]
electrode_field = np.zeros(skfem_mesh.nelements)
electrode_field[mat_of_el == "Au_sig"] = 1.0
electrode_field[mat_of_el == "Au_gnd"] = -1.0

skfem_mesh.plot(electrode_field, ax=ax0, shading="flat", cmap=plt.matplotlib.colors.ListedColormap(["#2c3e50", "#ecf0f1", "#e67e22"]))
ax0.text(GAP / 2 + 1.0, SLAB_H + 0.8, "SIGNAL (+1V)", color="white", fontweight="bold")
ax0.text(-GAP / 2 - 2.0, SLAB_H + 0.8, "GND", color="white", fontweight="bold")
ax0.set_title("Electrode Assignment: Signal vs Ground")
ax0.set_xlim([-8.0, 8.0])
ax0.set_ylim([-0.5, 2.5])
ax0.set_aspect("equal")

ax1 = axes[1]
skfem_mesh.plot(potential_nodes, ax=ax1, shading="gouraud", cmap="coolwarm", colorbar=True)
ax1.set_title(r"Anisotropic DC Potential $V(x, y)$ [V]")
ax1.set_xlim([-8.0, 8.0])
ax1.set_ylim([-0.5, 2.5])
ax1.set_aspect("equal")

plt.tight_layout()
plt.show()

In [ ]:
# %% [2] Optical Eigensolver (N1 x P1 Elements)
import scipy.constants
from skfem import ElementTriN1, ElementTriP0, ElementTriP1, BilinearForm, solve, condense
from skfem.helpers import curl, dot, grad
from skfem.utils import solver_eigen_scipy
from femwell.maxwell.waveguide import Mode, Modes, calculate_hfield, calculate_overlap

def compute_modes_anisotropic(
    basis_epsilon_r,
    eps_xx,
    eps_yy,
    eps_zz,
    wavelength,
    num_modes=8,
    n_guess=1.8755,
):
    c0 = scipy.constants.speed_of_light
    k0_val = 2.0 * np.pi / wavelength
    element = ElementTriN1() * ElementTriP1()
    basis = basis_epsilon_r.with_element(element)
    basis_eps = basis.with_element(basis_epsilon_r.elem)

    @BilinearForm(dtype=complex)
    def aform(e_t, e_z, v_t, v_z, w):
        return (
            curl(e_t) * curl(v_t) / k0_val**2
            - (w.eps_xx * e_t[0] * v_t[0] + w.eps_yy * e_t[1] * v_t[1])
            + dot(grad(e_z), v_t)
            + (w.eps_xx * e_t[0] * grad(v_z)[0] + w.eps_yy * e_t[1] * grad(v_z)[1])
            - w.eps_zz * e_z * v_z * k0_val**2
        )

    @BilinearForm(dtype=complex)
    def bform(e_t, e_z, v_t, v_z, w):
        return -dot(e_t, v_t) / k0_val**2

    A = aform.assemble(
        basis,
        eps_xx=basis_eps.interpolate(eps_xx),
        eps_yy=basis_eps.interpolate(eps_yy),
        eps_zz=basis_eps.interpolate(eps_zz),
    )
    B = bform.assemble(basis)

    bnd_dofs = basis.get_dofs(facets=skfem_mesh.boundary_facets())
    lams, xs = solve(
        *condense(-A, -B, D=bnd_dofs, x=basis.zeros(dtype=complex)),
        solver=solver_eigen_scipy(k=num_modes, sigma=k0_val**2 * n_guess**2),
    )

    xs[basis.split_indices()[1], :] /= 1j * np.sqrt(lams[np.newaxis, :] / k0_val**4)

    hs = []
    omega = k0_val * (c0 * 1e6)
    for i, lam in enumerate(lams):
        H = calculate_hfield(basis, xs[:, i], np.sqrt(lam), omega=omega)
        power = calculate_overlap(basis, xs[:, i], H, basis, xs[:, i], H)
        xs[:, i] /= np.sqrt(power)
        H /= np.sqrt(power)
        hs.append(H)

    return Modes(
        modes=[
            Mode(
                frequency=(c0 * 1e6) / wavelength,
                k=np.sqrt(lams[i]),
                basis_epsilon_r=basis_epsilon_r,
                epsilon_r=eps_xx,
                basis=basis,
                E=xs[:, i],
                H=hs[i],
            )
            for i in range(num_modes)
        ]
    )

eps_xx = np.array([MATERIALS[m]["eps_opt"][0] for m in mat_of_el], dtype=complex)
eps_yy = np.array([MATERIALS[m]["eps_opt"][1] for m in mat_of_el], dtype=complex)
eps_zz = np.array([MATERIALS[m]["eps_opt"][2] for m in mat_of_el], dtype=complex)

basis_epsilon_r = Basis(skfem_mesh, ElementTriP0(), intorder=4)

import time
t_eig0 = time.time()
modes = compute_modes_anisotropic(
    basis_epsilon_r,
    eps_xx,
    eps_yy,
    eps_zz,
    wavelength=wl,
    num_modes=num_modes_search,
    n_guess=n_guess,
)
print(f"Eigensolver complete in {time.time() - t_eig0:.1f}s: computed {len(modes)} optical modes around n_guess = {n_guess}.")

In [ ]:
# %% [3] High-Precision Diagnostics & Mode Gallery
from skfem import Functional

core_ids = [s_id for name, s_id in name_to_id.items() if name.split("___")[0] == "core"]
is_core = np.zeros(skfem_mesh.nelements, dtype=float)
is_core[np.isin(tri_subdomains, core_ids)] = 1.0
is_core_field = basis_p0.interpolate(is_core)

@Functional(dtype=complex)
def _px(w):
    return np.abs(w["E"][0][0]) ** 2

@Functional(dtype=complex)
def _ptot(w):
    return (
        np.abs(w["E"][0][0]) ** 2
        + np.abs(w["E"][0][1]) ** 2
        + np.abs(w["E"][1]) ** 2
    )

@Functional(dtype=complex)
def _p_core(w):
    return w["is_core"] * (
        np.abs(w["E"][0][0]) ** 2
        + np.abs(w["E"][0][1]) ** 2
        + np.abs(w["E"][1]) ** 2
    )

@Functional(dtype=complex)
def _ex_core_signed(w):
    return w["is_core"] * np.real(w["E"][0][0])

@Functional(dtype=complex)
def _ex_core_abs(w):
    return w["is_core"] * np.abs(np.real(w["E"][0][0]))

mode_diagnostics = []

print("=" * 82)
print(f"{'Idx':<4} | {'Re(neff)':<10} | {'Im(neff)':<12} | {'TE %':<8} | {'Core Pwr %':<10} | {'Unipolarity':<12}")
print("-" * 82)

for idx, mode in enumerate(modes):
    E_interp = mode.basis.interpolate(mode.E)

    P_x = np.real(_px.assemble(mode.basis, E=E_interp))
    P_tot = np.real(_ptot.assemble(mode.basis, E=E_interp))
    P_core = np.real(_p_core.assemble(mode.basis, E=E_interp, is_core=is_core_field))

    te_ratio = P_x / P_tot
    core_confinement = P_core / P_tot

    signed = np.real(_ex_core_signed.assemble(mode.basis, E=E_interp, is_core=is_core_field))
    absval = np.real(_ex_core_abs.assemble(mode.basis, E=E_interp, is_core=is_core_field))

    unipolarity = 0.0000 if absval < 1e-12 else np.abs(signed) / absval

    neff_val = complex(mode.k / k0)

    mode_diagnostics.append({
        "index": idx,
        "mode": mode,
        "neff": neff_val,
        "te_ratio": te_ratio,
        "core_conf": core_confinement,
        "unipolarity": unipolarity,
    })

    print(
        f"{idx:<4} | "
        f"{neff_val.real:<10.5f} | "
        f"{neff_val.imag:<12.2e} | "
        f"{te_ratio * 100:<7.2f}% | "
        f"{core_confinement * 100:<9.2f}% | "
        f"{unipolarity:<12.4f}"
    )

print("=" * 82)

# Full Mode Gallery Plot
cols = 4
rows = int(np.ceil(len(modes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(16, 3.4 * rows))
axes = np.array(axes).flatten()

for idx, diag in enumerate(mode_diagnostics):
    ax = axes[idx]
    E_interp = diag["mode"].basis.interpolate(diag["mode"].E)
    Ex_elem = np.mean(np.abs(E_interp[0].value[0]) ** 2, axis=-1)

    skfem_mesh.plot(Ex_elem, ax=ax, shading="flat", cmap="inferno")
    title_str = (
        f"Mode {idx}: n={diag['neff'].real:.4f}\n"
        f"TE={diag['te_ratio']*100:.1f}% | Conf={diag['core_conf']*100:.1f}%\n"
        f"Uni={diag['unipolarity']:.4f}"
    )
    ax.set_title(title_str, fontsize=9)
    ax.set_xlim([-4.0, 4.0])
    ax.set_ylim([-0.5, 1.5])
    ax.set_aspect("equal")

for j in range(len(modes), len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [ ]:
# %% [4] Overlap Integral, Vpi*L & Optical Attenuation
from skfem import Functional

# Fundamental TE mode identification (highest index matching modal filters)
guided_cands = [
    d for d in mode_diagnostics
    if d["te_ratio"] >= 0.75 and d["core_conf"] >= 0.20 and d["unipolarity"] >= 0.85   # same filter as the etched Lisbon-lifted notebook
]

if not guided_cands:
    raise RuntimeError("Fundamental TE rib mode not found. Check n_guess or eigensolver outputs.")

guided_cands.sort(key=lambda x: x["neff"].real, reverse=True)
fund_diag = guided_cands[0]
chosen_idx = fund_diag["index"]
fund_mode = fund_diag["mode"]
n_eff = fund_diag["neff"]

# 1. Optical Attenuation (Ohmic Metal Loss matching COMSOL Att_Opt)
att_opt_dB_m = np.abs(20.0 * np.log10(np.e) * k0_m * np.imag(n_eff))
att_opt_dB_cm = att_opt_dB_m / 100.0

# 2. Electro-Optic Overlap Integral (Gamma)
ln_mask = (mat_of_el == "LN")
Ex_dc_ln = np.zeros(skfem_mesh.nelements, dtype=float)
Ex_dc_ln[ln_mask] = Ex_dc[ln_mask]
Ex_dc_ln_field = basis_p0.interpolate(Ex_dc_ln)

E_fund_interp = fund_mode.basis.interpolate(fund_mode.E)

@Functional(dtype=complex)
def _num(w): 
    return w["Ex_dc_ln"] * np.abs(w["E"][0][0]) ** 2

@Functional(dtype=complex)
def _den(w): 
    return np.abs(w["E"][0][0]) ** 2

num = np.real(_num.assemble(fund_mode.basis, E=E_fund_interp, Ex_dc_ln=Ex_dc_ln_field))
den = np.real(_den.assemble(fund_mode.basis, E=E_fund_interp))

overlap = (GAP / V_bias) * (num / den)
Vpi_L_m = (GAP * 1e-6 * wl_m) / (r33 * np.abs(overlap) * (ne**3) * 2.0)
Vpi_L_Vcm = Vpi_L_m * 100.0
Vpi = Vpi_L_Vcm / L_cm

# 3. Formatted Simulation Summary
print("=" * 55)
print("             ELECTRO-OPTIC SIMULATION RESULTS")
print("=" * 55)
print(f"Selected Mode Index:         {chosen_idx}")
print(f"Effective Index (neff):      {n_eff.real:.6f} + {n_eff.imag:.2e}j")
print(f"Optical Attenuation:         {att_opt_dB_cm:.4e} dB/cm  ({att_opt_dB_m:.4e} dB/m)")
print(f"Overlap Integral (Gamma):    {overlap:.4f}")
print(f"|Vpi * L|:                   {Vpi_L_Vcm:.4f} V*cm")
print(f"|Vpi| (for L = {L_cm} cm):        {Vpi:.4f} V")
print("=" * 55)